# Chẩn đoán khoảng lệch: 28.6 (của ta) so với 32.7 (paper)

**Cài đặt:** GPU T4 x2, Internet On. *Add Input* **cả hai**: (a) dataset video EK100 (`download-ek100`) và (b) output của lần chạy `kaggle_verify_ek100` (`verify-ek100`). Notebook tự dò đường dẫn, không cần sửa tay
(để dùng lại logits, không phải chạy lại 45 phút). Tổng thời gian khoảng 1 giờ, phần lớn là bước 5.

1. Phân tích logits đã có (CPU, vài giây): độ nhạy theo số GPU, bootstrap, nhãn xáo trộn, bf16, 5 video lệch fps.
2. Kiểm tra decord có trả đúng frame không (so với PyAV).
3. fp16 so với fp32 trên 256 clip.
4. Chạy lại toàn bộ theo giao thức chuẩn (`--anchor action_start`).

In [ ]:
BRANCH = "claude/vjepa2-verify-improve-dsfgtg"
VJEPA2_COMMIT = "204698b45b3712590f06245fbfba32d3be539812"  # version the code was checked against
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!git clone -q https://github.com/facebookresearch/vjepa2.git /kaggle/working/vjepa2 && cd /kaggle/working/vjepa2 && git checkout -q $VJEPA2_COMMIT
!pip install -q decord webdataset timm einops
import os
os.environ["VJEPA2_ROOT"] = "/kaggle/working/vjepa2"
R = "/kaggle/working/repo"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Checkpoints (~1.3GB encoder+predictor, probe) -> /tmp so they do not count toward the 20GB output limit
!mkdir -p /tmp/ckpt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/vitl.pt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/evals/ek100-vitl-256.pt
!ls -lh /tmp/ckpt

In [ ]:
# Official annotations (the full train csv is required: probe output classes are derived from it)
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /kaggle/working/ek100-ann
!ls /kaggle/input; find /kaggle/input -maxdepth 3 | head -30

In [ ]:
!pip install -q av
import glob, os
def find_video_root(root="/kaggle/input"):
    """Folder holding the EK100 videos: common parent of all .MP4 files among the attached inputs."""
    mp4 = glob.glob(os.path.join(root, "**", "*.MP4"), recursive=True)
    if not any(os.path.basename(p) == "P01_11.MP4" for p in mp4):
        raise RuntimeError("EK100 val videos not found: attach the video dataset via Add Input (e.g. download-ek100)")
    return os.path.commonpath(mp4)
VIDEO_ROOT = find_video_root()
print("VIDEO_ROOT =", VIDEO_ROOT)
# Output of the earlier kaggle_verify_ek100 run (attach it via Add Input): the folder holding logits/official
prev = [p for p in glob.glob("/kaggle/input/**/logits/official", recursive=True)]
if not prev:
    raise RuntimeError("logits/official not found: attach the output of the kaggle_verify_ek100 run via Add Input")
PREV = os.path.dirname(os.path.dirname(prev[0]))
print("PREV =", PREV)
!mkdir -p /kaggle/working/logits && cp -r $PREV/logits/official /kaggle/working/logits/
# Rebuild the clip index for the current video paths (deterministic: same clip_ids as the earlier run)
!python $R/scripts/prepare_ek100.py --video_root $VIDEO_ROOT \
    --train_csv /kaggle/working/ek100-ann/EPIC_100_train.csv \
    --val_csv /kaggle/working/ek100-ann/EPIC_100_validation.csv \
    --out_dir /kaggle/working/work --probe_videos \
    --video_info /kaggle/working/ek100-ann/EPIC_100_video_info.csv | grep -E "val_clips|out_of_range_clips|warnings" 

In [ ]:
# 1. Analysis of existing logits (CPU)
!python $R/scripts/diagnose_ek100.py --work_dir /kaggle/working/work --logits_dir /kaggle/working/logits/official \
    --out /kaggle/working/diagnose.json

In [ ]:
# 2. Does decord return the requested frames? (compared with PyAV, 40 random clips)
!python $R/scripts/check_decoding.py --work_dir /kaggle/working/work --num_clips 40 --out /kaggle/working/check_decoding.json 2>&1 | tail -3

In [ ]:
# 3. fp32 on the first 256 clips of shard 0, then compare with the fp16 logits
!python $R/scripts/eval_ek100.py --work_dir /kaggle/working/work --out_dir /kaggle/working/logits_fp32 \
    --vitl_ckpt /tmp/ckpt/vitl.pt --probe_ckpt /tmp/ckpt/ek100-vitl-256.pt \
    --device cuda:0 --shard_id 0 --num_shards 2 --dtype fp32 --max_clips 256 --save_every 64 2>&1 | grep -E "clip/s|error" | tail -2
!python $R/scripts/diagnose_ek100.py --work_dir /kaggle/working/work --logits_dir /kaggle/working/logits/official \
    --compare_logits_dir /kaggle/working/logits_fp32/official --bootstrap 20 --out /kaggle/working/diagnose_fp32.json | tail -9

In [ ]:
# 4. Standard EK100 protocol: clip ends 1s before the action START (~45 min on 2 GPUs)
import subprocess, time
def shard_cmd(gpu):
    run = (f"python {R}/scripts/eval_ek100.py --work_dir /kaggle/working/work --out_dir /kaggle/working/logits "
           f"--vitl_ckpt /tmp/ckpt/vitl.pt --probe_ckpt /tmp/ckpt/ek100-vitl-256.pt --anchor action_start "
           f"--device cuda:{gpu} --shard_id {gpu} --num_shards 2 --dtype fp16 --num_workers 2")
    return f"for i in 1 2 3 4 5; do {run} && break; echo \"[retry $i]\"; done"
procs = {g: subprocess.Popen(["bash", "-c", shard_cmd(g) + f" >> /kaggle/working/eval_start_gpu{g}.log 2>&1"]) for g in (0, 1)}
while any(p.poll() is None for p in procs.values()):
    time.sleep(120)
    for g in procs:
        lines = [l for l in open(f"/kaggle/working/eval_start_gpu{g}.log").read().splitlines()
                 if any(k in l for k in ("clip/s", "[error]", "[retry", "to go", "Traceback"))]
        print(f"gpu{g}: {lines[-1] if lines else 'loading model...'}", flush=True)
print("exit codes:", {g: p.returncode for g, p in procs.items()})

In [ ]:
!python $R/scripts/compute_metrics.py --work_dir /kaggle/working/work --logits_dir /kaggle/working/logits/action_start \
    --out /kaggle/working/verify_ek100_vitl_action_start.json
!python $R/scripts/compute_metrics.py --work_dir /kaggle/working/work --logits_dir /kaggle/working/logits/official \
    --out /kaggle/working/verify_ek100_vitl.json

Gửi lại toàn bộ output của các cell 1–5 (hoặc các file `diagnose*.json`, `check_decoding.json`, `verify_*.json`).